In [ ]:
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
import numpy as np
from shapely.geometry import mapping
from shapely import geometry
from scipy import ndimage
from shapely import wkb
from xgboost import XGBClassifier
import json
import sys
# append the path of the parent directory
sys.path.append("..")
from utils.dol import *


CACHE_DIR = "/app/data/datasets/debug/bush/cache"
MODEL_SAVE_FOLDER = "/app/data/datasets/debug/bush/models"
MODEL_SAVE_PATH = os.path.join(MODEL_SAVE_FOLDER,'dol_xgb_model_v1.1.json')
METADATA_SAVE_PATH = os.path.join(MODEL_SAVE_FOLDER,'dol_xgb_model_v1.1_metadata.json')

# reload model
xgb_model_reloaded = XGBClassifier()
xgb_model_reloaded.load_model(MODEL_SAVE_PATH)

# reload metadata
with open(METADATA_SAVE_PATH) as f:
    metadata = json.load(f)
xgb_model_reloaded.scale_pos_weight = metadata["scale_pos_weight"]
THRESHOLD = metadata["decision_threshold"]
metadata

In [ ]:
gdf_forests_zones = gpd.read_file('/app/data/datasets/debug/bush/sources/ocsge_forests_clean_types_v3.gpkg',driver='GPKG')
gdf_waters_zones = gpd.read_file('/app/data/datasets/debug/bush/sources/COURS_D_EAU.shp')
gdf_u_zone = gpd.read_file('/app/data/datasets/debug/bush/sources/34_zones_u.gpkg',driver='GPKG')
gdf_forests_zones = gdf_forests_zones[gdf_forests_zones.forest_type==1]

In [ ]:
gdf_ciblage_2026 = gpd.read_file('/app/data/datasets/debug/bush/ciblage_2026/selection_pc_2026.shp')
gdf_ciblage_2026.head(5)

In [ ]:
gdf_ciblage_2026.groupby(['insee_com','choix_ddtm']).size().reset_index()

In [ ]:
34056 - castelnau-de-guers - ok
34075 - cesseras - ok
34090 - cres - ok
34095 - fabregues - ok
34117 - graissessac - ok
34142 - lodeve - ok a controler que entier
34146 - lunel-viel - ok
34221 - puechabon - ok
34235 - rosis - ok
34242 - saint-bauzille-de-montmel - ok
34245 - saint-chinian - ok
34320 - vailhauques - ok
34334 - vieussan

In [ ]:
communes_test = [
    {'name':'castelnau-de-guers', 'geozone_code': 34056,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_castelnau-de-guers_34056_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_160.tif',
                                                                                       '/app/runs/aigle_aerial_yolov_2024_castelnau-de-guers_34056_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_161.tif',
                                                                                       '/app/runs/aigle_aerial_yolov_2024_castelnau-de-guers_34056_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_175.tif',
                                                                                       '/app/runs/aigle_aerial_yolov_2024_castelnau-de-guers_34056_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_176.tif',
                                                                                       '/app/runs/aigle_aerial_yolov_2024_castelnau-de-guers_34056_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_190.tif',
                                                                                       '/app/runs/aigle_aerial_yolov_2024_castelnau-de-guers_34056_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_191.tif']
     },
    {'name':'cesseras', 'geozone_code': 34075,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_cesseras_34075_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_11.tif',
                                                                                   '/app/runs/aigle_aerial_yolov_2024_cesseras_34075_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_12.tif',
                                                                                   '/app/runs/aigle_aerial_yolov_2024_cesseras_34075_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_13.tif',
                                                                                   '/app/runs/aigle_aerial_yolov_2024_cesseras_34075_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_21.tif',
                                                                                   '/app/runs/aigle_aerial_yolov_2024_cesseras_34075_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_22.tif',
                                                                                   '/app/runs/aigle_aerial_yolov_2024_cesseras_34075_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_23.tif']
     },
    {'name':'cres', 'geozone_code': 34090,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_cres_34090_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_272.tif',
                                                                             '/app/runs/aigle_aerial_yolov_2024_cres_34090_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_273.tif',
                                                                             '/app/runs/aigle_aerial_yolov_2024_cres_34090_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_282.tif',
                                                                             '/app/runs/aigle_aerial_yolov_2024_cres_34090_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_283.tif']},
    {'name':'fabregues', 'geozone_code': 34095,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_fabregues_34095_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_233.tif',
                                                                                   '/app/runs/aigle_aerial_yolov_2024_fabregues_34095_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_234.tif',
                                                                                   '/app/runs/aigle_aerial_yolov_2024_fabregues_34095_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_246.tif',
                                                                                   '/app/runs/aigle_aerial_yolov_2024_fabregues_34095_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_247.tif',
                                                                                   '/app/runs/aigle_aerial_yolov_2024_fabregues_34095_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_258.tif',
                                                                                   '/app/runs/aigle_aerial_yolov_2024_fabregues_34095_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_259.tif']},
    {'name':'graissessac', 'geozone_code': 34117,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_graissessac_34117_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_76.tif',
                                                                                '/app/runs/aigle_aerial_yolov_2024_graissessac_34117_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_90.tif']},
    {'name':'lodeve', 'geozone_code': 34142,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_lodeve_34142_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_135.tif']},
    {'name':'lunel-viel', 'geozone_code': 34146,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_lunel-viel_34146_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_290.tif',
                                                                                '/app/runs/aigle_aerial_yolov_2024_lunel-viel_34146_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_291.tif',
                                                                                '/app/runs/aigle_aerial_yolov_2024_lunel-viel_34146_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_297.tif',
                                                                                '/app/runs/aigle_aerial_yolov_2024_lunel-viel_34146_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_298.tif']},
    {'name':'puechabon', 'geozone_code': 34221,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_puechabon_34221_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_196.tif',
                                                                              '/app/runs/aigle_aerial_yolov_2024_puechabon_34221_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_197.tif',
                                                                              '/app/runs/aigle_aerial_yolov_2024_puechabon_34221_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_198.tif',
                                                                              '/app/runs/aigle_aerial_yolov_2024_puechabon_34221_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_210.tif',
                                                                              '/app/runs/aigle_aerial_yolov_2024_puechabon_34221_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_211.tif',
                                                                              '/app/runs/aigle_aerial_yolov_2024_puechabon_34221_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_212.tif',
                                                                              '/app/runs/aigle_aerial_yolov_2024_puechabon_34221_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_223.tif',
                                                                              '/app/runs/aigle_aerial_yolov_2024_puechabon_34221_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_224.tif',
                                                                              '/app/runs/aigle_aerial_yolov_2024_puechabon_34221_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_225.tif']},
    {'name':'rosis', 'geozone_code': 34235,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_rosis_34235_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_63.tif',
                                                                          '/app/runs/aigle_aerial_yolov_2024_rosis_34235_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_64.tif',
                                                                          '/app/runs/aigle_aerial_yolov_2024_rosis_34235_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_65.tif',
                                                                          '/app/runs/aigle_aerial_yolov_2024_rosis_34235_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_74.tif',
                                                                          '/app/runs/aigle_aerial_yolov_2024_rosis_34235_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_75.tif',
                                                                          '/app/runs/aigle_aerial_yolov_2024_rosis_34235_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_76.tif']},
    {'name':'saint-bauzille-de-montmel', 'geozone_code': 34242,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_saint-bauzille-de-montmel_34242_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_274.tif',
                                                                                                 '/app/runs/aigle_aerial_yolov_2024_saint-bauzille-de-montmel_34242_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_275.tif',
                                                                                                 '/app/runs/aigle_aerial_yolov_2024_saint-bauzille-de-montmel_34242_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_276.tif',
                                                                                                 '/app/runs/aigle_aerial_yolov_2024_saint-bauzille-de-montmel_34242_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_284.tif',
                                                                                                 '/app/runs/aigle_aerial_yolov_2024_saint-bauzille-de-montmel_34242_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_285.tif']},
    {'name':'saint-chinian', 'geozone_code': 34245,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_saint-chinian_34245_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_49.tif',
                                                                                     '/app/runs/aigle_aerial_yolov_2024_saint-chinian_34245_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_50.tif',
                                                                                     '/app/runs/aigle_aerial_yolov_2024_saint-chinian_34245_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_51.tif',
                                                                                     '/app/runs/aigle_aerial_yolov_2024_saint-chinian_34245_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_58.tif',
                                                                                     '/app/runs/aigle_aerial_yolov_2024_saint-chinian_34245_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_59.tif',
                                                                                     '/app/runs/aigle_aerial_yolov_2024_saint-chinian_34245_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_60.tif']},
    {'name':'vailhauques', 'geozone_code': 34320,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_vailhauques_34320_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_236.tif',
                                                                                  '/app/runs/aigle_aerial_yolov_2024_vailhauques_34320_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_237.tif',
                                                                                  '/app/runs/aigle_aerial_yolov_2024_vailhauques_34320_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_249.tif',
                                                                                  '/app/runs/aigle_aerial_yolov_2024_vailhauques_34320_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_250.tif']},
    {'name':'vieussan', 'geozone_code': 34334,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_vieussan_34334_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_52.tif',
                                                                             '/app/runs/aigle_aerial_yolov_2024_vieussan_34334_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_53.tif',
                                                                             '/app/runs/aigle_aerial_yolov_2024_vieussan_34334_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_61.tif',
                                                                             '/app/runs/aigle_aerial_yolov_2024_vieussan_34334_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_62.tif',
                                                                             '/app/runs/aigle_aerial_yolov_2024_vieussan_34334_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_72.tif',
                                                                             '/app/runs/aigle_aerial_yolov_2024_vieussan_34334_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_73.tif']}
]
imgs_bounds = []
for comm in communes_test :
    for img_path in comm['result_segmentation_files']:
        with rasterio.open(img_path) as src :
            bbox = src.bounds
            bbox_polygon = geometry.box(*bbox)
            print(bbox_polygon)
            imgs_bounds.append([img_path, bbox_polygon])

gdf_img = gpd.GeoDataFrame(data= imgs_bounds, columns=['image_path','geometry'], geometry='geometry', crs='EPSG:2154')
gdf_img.to_crs('EPSG:2154',inplace=True)
gdf_img.drop_duplicates(subset='image_path',inplace=True)

In [ ]:
gdf_zone_data = gdf_ciblage_2026[gdf_ciblage_2026.insee_com.isin([str(x['geozone_code']) for x in communes_test])]
gdf_zone_data = gdf_zone_data[['insee_com','id_ilot','choix_ddtm','geometry']]
gdf_zone_data

In [ ]:
gdf_zone_data = gpd.sjoin(gdf_img,gdf_zone_data, how='right', predicate='intersects').drop(columns='index_left')
gdf_zone_data = gdf_zone_data[~gdf_zone_data.image_path.isna()]
gdf_zone_data.rename(columns={'geometry':'geom'}, inplace=True)
#gdf_zone_data.set_geometry("geometry",inplace=True)
gdf_zone_data

In [ ]:
x_ml_features, x_business_features =  preprocess_features(gdf_zone_data, gdf_forests_zones, gdf_waters_zones, gdf_u_zone, cache_dir = CACHE_DIR, debug=False)
x_ml_features

In [ ]:
y_proba = xgb_model_reloaded.predict_proba(x_ml_features)[:, 1]

#y_pred_custom = (y_proba >= THRESHOLD)
gdf_zone_data_debug["proba_control"] = y_proba
gdf_zone_data_debug["pred_control"] = 0
gdf_zone_data_debug.loc[gdf_zone_data_debug["proba_control"] >= THRESHOLD-0.15,"pred_control"] = 1
gdf_zone_data_debug

In [ ]:
gdf_zone_data.set_geometry("geom",inplace=True)

In [ ]:
test_results_gdf = postprocess_pred_control(gdf_zone_data, x_business_features)

test_results_gdf

In [ ]:
test_results_gdf.to_file(
    "/app/data/datasets/debug/bush/pred_ciblage_2026_dol_zones_xgb_v1.1.gpkg",
    driver="GPKG"
)